In [1]:
!pip install faiss-cpu sentence-transformers rank_bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 92.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 113.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 98.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 34.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 15.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 6.4 MB/s eta 0:00:000:00:0100:01
  Attempting un

In [2]:
import os
import pickle
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, StoppingCriteria, StoppingCriteriaList
from peft import PeftModel
import time
import gc

2025-11-29 01:52:39.959847: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764381160.127893      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764381160.178284      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [3]:
class VietnameseRAGSystem:
    def __init__(self, embedding_model_name="intfloat/multilingual-e5-large-instruct", cross_encoder_name="namdp-ptit/ViRanker", cache_dir="/kaggle/working", device=None):
        if device is None:
            device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.device = device

        print("Loading embedding model...")
        self.embedding_model = SentenceTransformer(embedding_model_name, device=device)
        self.embedding_model.eval()
        
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        print("Loading cross-encoder model...")
        self.cross_encoder = CrossEncoder(cross_encoder_name, device=device)

        self.index = None
        self.chunks = []
        self.metadata = []
        self.bm25_index = None
        self.normalized_texts = []
        self.cache_dir = cache_dir
        os.makedirs(cache_dir, exist_ok=True)
       
        self.retrieval_task = 'Given a query about Vietnamese history, retrieve relevant historical passages that answer the query'
   
    def normalize_text(self, text):
        return text.lower()
   
    def get_detailed_instruct(self, query):
        return f'Instruct: {self.retrieval_task}\\\\nQuery: {query}'
   
    def load_index(self, faiss_file, metadata_file):
        self.index = faiss.read_index(faiss_file)
       
        with open(metadata_file, 'rb') as f:
            index_data = pickle.load(f)
       
        self.chunks = index_data['chunks']
        self.metadata = index_data['metadata']
        self.normalized_texts = index_data['normalized_texts']
       
        try:
            if len(self.normalized_texts) > 0:
                tokenized_corpus = [re.sub(r'[.,!?;:\\\"()]+', ' ', text).split() for text in self.normalized_texts]
                self.bm25_index = BM25Okapi(tokenized_corpus)
                print("BM25 index created successfully")
            else:
                print("Empty corpus, skip BM25")
                self.bm25_index = None
        except Exception as e:
            print(f"Error creating BM25: {e}. Continue without BM25.")
            self.bm25_index = None
       
        print(f"Loaded index with {len(self.chunks)} chunks")
   
    def hybrid_search(self, query, top_k=5, filter_trieu_dai=None, filter_chu_de=None):
        if self.index is None:
            raise ValueError("Index not initialized!")
       
        initial_results = self._semantic_search(
            query,
            top_k=min(50, len(self.chunks)),
            filter_trieu_dai=filter_trieu_dai,
            filter_chu_de=filter_chu_de
        )
       
        if not initial_results:
            return []
       
        reranked_results = self._rerank_with_cross_encoder(query, initial_results, top_k=top_k)
       
        return reranked_results
    
    def _semantic_search(self, query, top_k=10, filter_trieu_dai=None, filter_chu_de=None):
        instructed_query = self.get_detailed_instruct(query)
        query_embedding = self.embedding_model.encode([instructed_query], normalize_embeddings=True)
        
        candidate_size = min(top_k * 3, len(self.chunks))
        semantic_scores, semantic_indices = self.index.search(query_embedding, candidate_size)
        
        results = []
        for score, idx in zip(semantic_scores[0], semantic_indices[0]):
            if idx < len(self.chunks) and score > 0.1:
                chunk = self.chunks[idx]
                metadata = self.metadata[idx]
                
                if filter_trieu_dai and metadata.get('trieu_dai') != filter_trieu_dai:
                    continue
                if filter_chu_de and metadata.get('chu_de') != filter_chu_de:
                    continue
                
                results.append({
                    'text': chunk['text'],
                    'metadata': metadata,
                    'score': float(score),
                    'original_index': idx
                })
                
                if len(results) >= top_k:
                    break
        
        return results
   
    def _rerank_with_cross_encoder(self, query, initial_results, top_k=5):
        if not initial_results:
            return []
       
        max_text_length = 512
        documents = [result['text'][:max_text_length] for result in initial_results]
        pairs = [[query, doc] for doc in documents]
       
        print(f"Re-ranking {len(pairs)} results...")
        try:
            ce_scores = self.cross_encoder.predict(pairs, batch_size=32)
        except Exception as e:
            print(f"Cross-encoder error: {e}. Use initial results.")
            return initial_results[:top_k]
       
        final_results = []
        for i, (ce_score, original_result) in enumerate(zip(ce_scores, initial_results)):
            semantic_score = original_result['score']
            combined_score = 0.7 * ce_score + 0.3 * semantic_score
           
            final_results.append({
                'text': original_result['text'],
                'metadata': original_result['metadata'],
                'score': float(combined_score),
                'ce_score': float(ce_score),
                'original_score': semantic_score,
                'reranked': True
            })
       
        final_results.sort(key=lambda x: x['score'], reverse=True)
        return final_results[:top_k]
   
    def search(self, query, top_k=5, filter_trieu_dai=None, filter_chu_de=None):
        return self.hybrid_search(query, top_k, filter_trieu_dai, filter_chu_de)
   
    def get_available_filters(self):
        trieu_dais = set(m.get('trieu_dai') for m in self.metadata if m.get('trieu_dai'))
        chu_des = set(m.get('chu_de') for m in self.metadata if m.get('chu_de'))
       
        return {
            'trieu_dai': sorted(list(trieu_dais)),
            'chu_de': sorted(list(chu_des))
        }

In [4]:
rag_system = VietnameseRAGSystem()
faiss_cache_file = "/kaggle/input/vectordb/rag_index.faiss"
metadata_cache_file = "/kaggle/input/vectordb/rag_metadata.pkl"

rag_system.load_index(faiss_cache_file, metadata_cache_file)

filters = rag_system.get_available_filters()
print(f"RAG system ready with {len(rag_system.chunks)} documents")
print(f"Available dynasties: {len(filters['trieu_dai'])}")
print(f"Available topics: {len(filters['chu_de'])}")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_xlm-roberta_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

BM25 index created successfully
Loaded index with 19509 chunks
RAG system ready with 19509 documents
Available dynasties: 1118
Available topics: 18882


In [5]:
model_id = "Qwen/Qwen3-4B"
adapter_dir = "/kaggle/input/qwen-finetuned/transformers/default/3/qwen_finetuned"

import os
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

tokenizer = AutoTokenizer.from_pretrained(adapter_dir)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
    use_cache=True,
    low_cpu_mem_usage=True
)

model = PeftModel.from_pretrained(base_model, adapter_dir)
model = model.merge_and_unload()
model.eval()

if hasattr(model, 'config'):
    model.config.use_cache = True

print("Model and tokenizer loaded successfully")
print(f"Model device: {next(model.parameters()).device}")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model and tokenizer loaded successfully
Model device: cuda:0


In [47]:
MAX_CLASSIFICATION_TOKENS = 8
MAX_RESPONSE_TOKENS = 512
MIN_RESPONSE_LENGTH = 20

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def force_memory_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        for _ in range(3):
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
        time.sleep(0.1)
        gc.collect()
        torch.cuda.empty_cache()

class EndStoppingCriteria(StoppingCriteria):
    def __init__(self, tokenizer, end_sequence="(end)"):
        self.tokenizer = tokenizer
        self.end_sequence = end_sequence.lower()
        self.end_length = len(end_sequence)
    
    def __call__(self, input_ids, scores, **kwargs):
        if input_ids.shape[1] < self.end_length:
            return False
        
        recent_tokens = input_ids[0][-self.end_length:]
        recent_text = self.tokenizer.decode(recent_tokens, skip_special_tokens=True).lower()
        
        return self.end_sequence in recent_text

class QuestionClassifier:
    def __init__(self, tokenizer, model, device):
        self.tokenizer = tokenizer
        self.model = model
        self.device = device
        self.classification_cache = {}
    
    def classify_question(self, question: str) -> str:
        if question in self.classification_cache:
            return self.classification_cache[question]
            
        classification_prompt = f"""PHÂN LOẠI CÂU HỎI: Chọn MỘT trong 4 loại dưới đây:
            
1. HISTORY_DIRECT: Câu hỏi CỤ THỂ về sự kiện, nhân vật, thời gian lịch sử Việt Nam (có chi tiết rõ ràng)
2. OUT_OF_DOMAIN: Câu hỏi HOÀN TOÀN KHÔNG LIÊN QUAN đến lịch sử/văn hóa Việt Nam (thời tiết, thể thao, ẩm thực, công nghệ hiện đại)
3. INSUFFICIENT_INFO: Câu hỏi về lịch sử Việt Nam nhưng THÔNG TIN KHÔNG TỒN TẠI (ví dụ: công nghệ hiện đại trong thời phong kiến, phát minh không có thật)
4. VAGUE: Câu hỏi QUÁ RỘNG, MƠ HỒ, THIẾU CHI TIẾT (cần làm rõ)

QUAN TRỌNG: 
- INSUFFICIENT_INFO: vẫn là câu hỏi về lịch sử, nhưng thông tin không có thật
- OUT_OF_DOMAIN: không phải câu hỏi về lịch sử

VÍ DỤ:
- "Vua Quang Trung đánh quân Thanh năm nào?" -> HISTORY_DIRECT  
- "Thời tiết Hà Nội thế nào?" -> OUT_OF_DOMAIN
- "Nhà Trần có dùng điện thoại không?" -> INSUFFICIENT_INFO (về lịch sử nhưng không có thật)
- "Triều Nguyễn có Internet không?" -> INSUFFICIENT_INFO  
- "Kể về lịch sử Việt Nam" -> VAGUE
- "Cristiano Ronaldo là ai?" -> OUT_OF_DOMAIN
- "Thời kỳ Đổi mới bắt đầu từ năm nào?" -> HISTORY_DIRECT
- "Trận Điện Biên Phủ diễn ra trong thời gian nào?" -> HISTORY_DIRECT

CÂU HỎI CẦN PHÂN LOẠI: {question}

CHỈ TRẢ LỜI MỘT TỪ: HISTORY_DIRECT, OUT_OF_DOMAIN, INSUFFICIENT_INFO, hoặc VAGUE
Kết quả:"""

        inputs = self.tokenizer(
            classification_prompt, 
            return_tensors="pt", 
            truncation=True, 
            max_length=1024,
            padding=True
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        try:
            with torch.inference_mode():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=MAX_CLASSIFICATION_TOKENS,
                    do_sample=False,
                    pad_token_id=self.tokenizer.eos_token_id,
                    eos_token_id=self.tokenizer.eos_token_id
                )

            resp = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            if resp.startswith(classification_prompt):
                classification = resp[len(classification_prompt):].strip()
            else:
                if "Kết quả:" in resp:
                    classification = resp.split("Kết quả:")[-1].strip()
                else:
                    classification = resp.strip()

            first_token = classification.split()[0] if classification else ""
            candidate = first_token.upper().strip().strip('":,.')
            
            if "HISTORY" in candidate:
                result = "HISTORY_DIRECT"
            elif "OUT" in candidate or "DOMAIN" in candidate:
                result = "OUT_OF_DOMAIN"
            elif "INSUFFICIENT" in candidate or "INSUFFICIENT_INFO" in candidate:
                result = "INSUFFICIENT_INFO"
            elif "VAGUE" in candidate:
                result = "VAGUE"
            else:
                c_up = classification.upper()
                if "HISTORY_DIRECT" in c_up:
                    result = "HISTORY_DIRECT"
                elif "OUT_OF_DOMAIN" in c_up or "OUT-OF-DOMAIN" in c_up:
                    result = "OUT_OF_DOMAIN"
                elif "INSUFFICIENT" in c_up:
                    result = "INSUFFICIENT_INFO"
                elif "VAGUE" in c_up:
                    result = "VAGUE"
                else:
                    result = "HISTORY_DIRECT"

            self.classification_cache[question] = result
            return result
            
        finally:
            del inputs, outputs
            clear_memory()

class QASystemWithClassification:
    def __init__(self, tokenizer, model, device, rag_system):
        self.tokenizer = tokenizer
        self.model = model
        self.device = device
        self.rag_system = rag_system
        self.classifier = QuestionClassifier(tokenizer, model, device)
        
        self.model.eval()
        self._question_count = 0
    
    def generate_answer(self, question: str) -> str:
        self._question_count += 1
        
        if self._question_count % 2 == 0:
            force_memory_cleanup()
            
        try:
            question_type = self.classifier.classify_question(question)
            
            if question_type == "OUT_OF_DOMAIN":
                return self._handle_out_of_domain(question)
            elif question_type == "VAGUE":
                return self._handle_vague_question(question)
            elif question_type == "INSUFFICIENT_INFO":
                return self._handle_insufficient_info(question)
            else:
                return self._handle_history_question(question)
        finally:
            clear_memory()
    
    def _handle_out_of_domain(self, question: str) -> str:
        response_prompt = f"""Bạn là trợ lý chuyên gia về lịch sử Việt Nam. Hãy trả lời câu hỏi sau BẰNG TIẾNG VIỆT CÓ DẤU:

Câu hỏi: {question}

Đây là câu hỏi KHÔNG thuộc phạm vi lịch sử Việt Nam.

YÊU CẦU:
✅ Lịch sự từ chối: "Tôi rất tiếc, nhưng câu hỏi này không thuộc lĩnh vực chuyên môn của tôi."
✅ Giải thích rõ: Tôi chỉ hỗ trợ các câu hỏi về lịch sử và văn hóa Việt Nam từ thời cổ đại đến hiện đại.
✅ Đề xuất ÍT NHẤT 4 chủ đề lịch sử phong phú, cụ thể, hấp dẫn, ví dụ:
   • Các triều đại phong kiến: Lý, Trần, Lê, Nguyễn
   • Các cuộc kháng chiến chống ngoại xâm: chống Nguyên Mông, Minh, Thanh, Pháp, Mỹ
   • Thời kỳ Đổi mới (1986–nay) và cải cách kinh tế
   • Văn hóa, tín ngưỡng, kiến trúc qua các thời kỳ (Chăm Pa, Đại Việt, Huế…)
✅ Khuyến khích người dùng đặt câu hỏi lịch sử cụ thể
✅ ĐỘ DÀI TỐI THIỂU: 120 từ

Kết thúc bằng "(end)"

Trả lời:"""
        
        return self._generate_response(response_prompt)
        
    def _handle_vague_question(self, question: str) -> str:
        response_prompt = f"""Bạn là trợ lý chuyên gia về lịch sử Việt Nam. Hãy trả lời câu hỏi sau BẰNG TIẾNG VIỆT CÓ DẤU:

Câu hỏi: {question}

Đây là câu hỏi QUÁ RỘNG, cần được làm rõ để có thể trả lời chính xác.

YÊU CẦU:
✅ Giải thích: "Câu hỏi này rất rộng và bao quát nhiều giai đoạn/sự kiện khác nhau."
✅ Nêu lý do: Lịch sử Việt Nam kéo dài hơn 4000 năm, với hàng trăm sự kiện quan trọng → cần thu hẹp phạm vi.
✅ Đề nghị người dùng cung cấp: thời kỳ cụ thể (ví dụ: thời Lý, thế kỷ 10?), nhân vật, sự kiện, hoặc khía cạnh (chính trị, văn hóa, quân sự?).
✅ Đưa ra 4 VÍ DỤ CỤ THỂ về cách hỏi tốt:
   - "Chính sách cai trị của nhà Nguyễn đối với dân tộc thiểu số như thế nào?"
   - "Kinh tế thời Lê Sơ phát triển ra sao?"
   - "Vai trò của phụ nữ trong khởi nghĩa Hai Bà Trưng?"
   - "Ảnh hưởng của Nho giáo dưới triều Lý là gì?"
✅ ĐỘ DÀI TỐI THIỂU: 120 từ. Thái độ tích cực, hỗ trợ.

Kết thúc bằng "(end)"

Trả lời:"""
        
        return self._generate_response(response_prompt)
    
    def _handle_insufficient_info(self, question: str) -> str:
        context = self._get_context_for_question(question)
        if context:
            response_prompt = f"""Bạn là chuyên gia lịch sử Việt Nam. Hãy trả lời câu hỏi sau BẰNG TIẾNG VIỆT CÓ DẤU:

THÔNG TIN THAM KHẢO:
{context}

Câu hỏi: {question}

LƯU Ý: Đây là câu hỏi dựa trên giả định KHÔNG TỒN TẠI trong lịch sử (ví dụ: công nghệ hiện đại trong thời phong kiến).

YÊU CẦU BẮT BUỘC:
1. KHẲNG ĐỊNH NGAY: "Câu hỏi này đề cập đến một yếu tố KHÔNG CÓ THẬT trong lịch sử Việt Nam."
2. PHÂN TÍCH THỜI KỲ: Nêu triều đại/thế kỷ, trình độ kỹ thuật – xã hội thời đó
3. SO SÁNH LỊCH SỬ: Khi nào yếu tố đó (điện thoại, Internet...) mới xuất hiện trên thế giới?
4. THÔNG TIN THAY THẾ: Người ta dùng gì để liên lạc/giao tiếp vào thời đó?
5. ĐỀ XUẤT 3 CÂU HỎI HỢP LÝ: Ví dụ: "Phương tiện truyền tin thời nhà Trần là gì?", "Hệ thống dịch trạm thời Lý hoạt động ra sao?"
6. ĐỘ DÀI: TỐI THIỂU 150 từ. Viết thành đoạn văn mạch lạc.

❌ Tuyệt đối KHÔNG dùng từ như "có thể", "có lẽ", "không rõ" → bạn là chuyên gia, hãy khẳng định dựa trên sử học
✅ Kết thúc bằng "(end)"

Trả lời:"""
        else:
            response_prompt = f"""Bạn là chuyên gia lịch sử Việt Nam. Hãy trả lời câu hỏi sau BẰNG TIẾNG VIỆT CÓ DẤU:

Câu hỏi: {question}

ĐÂY LÀ CÂU HỎI DỰA TRÊN GIẢ ĐỊNH PHI LỊCH SỬ.

YÊU CẦU CHI TIẾT:
- BẮT ĐẦU: "Câu hỏi này không phản ánh thực tế lịch sử."
- GIẢI THÍCH RÕ: Thời nhà Trần (1225–1400) là thời kỳ trung đại, chưa có điện, máy móc, chứ đừng nói điện thoại.
- NÊU DẪN CHỨNG: Điện thoại do Alexander Graham Bell phát minh năm 1876 tại Mỹ — cách thời Trần hơn 450 năm.
- THÔNG TIN THỰC TẾ: Thời Trần dùng trạm dịch, khói lửa, trống, hoặc sứ giả để truyền tin.
- ĐỀ XUẤT 4 CÂU HỎI THAY THẾ HỮU ÍCH về lịch sử Việt Nam thời Trần.
- ĐỘ DÀI TỐI THIỂU: 150 từ. Viết mạch lạc, học thuật, không vòng vo.

✅ Tuyệt đối không né tránh bằng câu ngắn.
✅ Kết thúc bằng "(end)"

Trả lời:"""
        
        return self._generate_response(response_prompt)
    
    def _handle_history_question(self, question: str) -> str:
        context = self._get_context_for_question(question)
        if context:
            response_prompt = f"""Bạn là chuyên gia lịch sử Việt Nam. Hãy trả lời câu hỏi sau BẰNG TIẾNG VIỆT:
    
    DỮ LIỆU LỊCH SỬ TỪ HỆ THỐNG:
    {context}
    
    Câu hỏi: {question}
    
    YÊU CẦU NGHIÊM NGẶT:
    ✅ Trả lời ĐẦY ĐỦ, CHI TIẾT, CHÍNH XÁC dựa HOÀN TOÀN trên dữ liệu trên
    ✅ Cung cấp BỐI CẢNH LỊCH SỬ đầy đủ cho câu trả lời
    ✅ Mô tả DIỄN BIẾN, NGUYÊN NHÂN, KẾT QUẢ nếu có trong dữ liệu
    ✅ Giải thích Ý NGHĨA LỊCH SỬ của sự kiện/nhân vật
    ✅ Cung cấp thông tin PHỤ TRỢ liên quan từ dữ liệu
    ✅ Sắp xếp thông tin LOGIC, RÕ RÀNG, DỄ HIỂU
    ✅ Ưu tiên thông tin CHI TIẾT và TOÀN DIỆN nhất
    
    ❌ Nếu dữ liệu KHÔNG ĐỦ thông tin, hãy THỪA NHẬN RÕ RÀNG
    ❌ TUYỆT ĐỐI KHÔNG thêm bất kỳ thông tin nào không có trong dữ liệu
    ❌ TUYỆT ĐỐI KHÔNG sử dụng tiếng Anh trong câu trả lời
    
    Hãy trả lời bằng TIẾNG VIỆT có dấu đầy đủ, cung cấp thông tin CHI TIẾT, ĐẦY ĐỦ NGỮ CẢNH và TOÀN DIỆN nhất có thể. Kết thúc bằng (end)
    
    Trả lời:"""
        else:
            response_prompt = f"""Bạn là chuyên gia lịch sử Việt Nam. Hãy trả lời câu hỏi sau BẰNG TIẾNG VIỆT:
    
    Câu hỏi: {question}
    
    THÔNG BÁO: Không tìm thấy dữ liệu phù hợp trong hệ thống.
    
    YÊU CẦU:
    ✅ Thông báo LỊCH SỰ về việc không có đủ thông tin để trả lời
    ✅ Giải thích CHI TIẾT có thể lý do không tìm thấy dữ liệu
    ✅ Đề xuất 4-5 chủ đề lịch sử Việt Nam CỤ THỂ mà người dùng có thể hỏi
    ✅ Cung cấp ví dụ về các câu hỏi CHI TIẾT hơn
    ✅ Giữ thái độ HỖ TRỢ và CHUYÊN NGHIỆP
    
    Hãy cung cấp câu trả lời ĐẦY ĐỦ, HỮU ÍCH bằng tiếng Việt. Kết thúc bằng (end)
    
    Trả lời:"""
        
        return self._generate_response(response_prompt)
    
    def _get_context_for_question(self, question: str, top_k: int = 5) -> str:
        results = self.rag_system.search(question, top_k=50)
        
        if not results:
            return None

        results = sorted(results, key=lambda x: x.get('score', 0), reverse=True)
        top_results = results[:min(5, len(results))]
        
        context_parts = []
        for i, result in enumerate(top_results, 1):
            text = result['text'].strip()
            context_parts.append(f"{text}")

        return "\n\n".join(context_parts)
    
    def _generate_response(self, prompt: str) -> str:
        inputs = None
        
        try:
            inputs = self.tokenizer(
                prompt, 
                return_tensors="pt", 
                truncation=True, 
                max_length=2048,
                padding=True
            )
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
            
            stopping_criteria = StoppingCriteriaList([EndStoppingCriteria(self.tokenizer)])
            
            with torch.inference_mode():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=MAX_RESPONSE_TOKENS,
                    do_sample=True,
                    temperature=0.5,
                    pad_token_id=self.tokenizer.eos_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                    stopping_criteria=stopping_criteria
                )
            
            full_output = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            
            if full_output.startswith(prompt):
                response = full_output[len(prompt):].strip()
            else:
                response = full_output.strip()
            
            response = re.split(r'\s*\(end\)\s*', response, flags=re.IGNORECASE)[0].strip()
            response = re.sub(r'\s+', ' ', response).strip()
            
            if len(response) < MIN_RESPONSE_LENGTH:
                return "Tôi không tìm thấy thông tin cụ thể trong cơ sở dữ liệu để trả lời đầy đủ cho câu hỏi này."
            
            return response
            
        except Exception:
            return "Xin lỗi, tôi gặp sự cố khi xử lý câu hỏi này."
        finally:
            if inputs is not None:
                del inputs
            clear_memory()

In [48]:
device = next(model.parameters()).device

force_memory_cleanup()
clear_memory()

qa_system = QASystemWithClassification(tokenizer, model, device, rag_system)
print("Hệ thống đã khởi tạo thành công")

test_questions = [
    "Vua Quang Trung tên thật là gì?",
    "Thời kỳ Đổi mới bắt đầu từ năm nào?",
    "Trận Điện Biên Phủ diễn ra trong thời gian nào?",
    "Thời tiết Hà Nội hôm nay thế nào?",
    "Nhà Trần có dùng điện thoại không?"
]

for question in test_questions:
    print(f"\nCâu hỏi: {question}")
    start_time = time.time()
    answer = qa_system.generate_answer(question)
    total_time = time.time() - start_time
    print(f"Thời gian: {total_time:.2f}s")
    print(f"Trả lời: {answer}")
    time.sleep(2.0)

force_memory_cleanup()
print("\nTest hoàn tất")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Hệ thống đã khởi tạo thành công

Câu hỏi: Vua Quang Trung tên thật là gì?
Re-ranking 50 results...
Thời gian: 12.32s
Trả lời: Vua Quang Trung tên thật là Nguyễn Huệ. Ông là vị tướng lĩnh kiệt xuất của triều đại Tây Sơn, đã có công lớn trong việc dẹp loạn, đánh đuổi quân xâm lược và thống nhất đất nước. Sau khi lên ngôi Hoàng đế, ông đã tiến hành nhiều cải cách quan trọng, xây dựng lại đất nước và ban hành nhiều chính sách táo bạo. Tuy nhiên, ông đã qua đời sớm vào năm 1792, để lại nhiều vấn đề chưa giải quyết, đặc biệt là việc đòi lại đất đai bị chiếm bởi quân Thanh.

Câu hỏi: Thời kỳ Đổi mới bắt đầu từ năm nào?


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Re-ranking 50 results...
Thời gian: 11.48s
Trả lời: Thời kỳ Đổi mới bắt đầu từ năm 1986, khi Đại hội VI của Đảng Cộng sản Việt Nam xác định rõ đường lối đổi mới. Bối cảnh quốc tế lúc bấy giờ với sự thành công của cải cách mở cửa ở Trung Quốc và xu hướng hợp tác thay thế đối đầu đã tạo điều kiện thuận lợi. Sự kiện này đánh dấu bước ngoặt quan trọng trong lịch sử Việt Nam, mở ra một kỷ nguyên mới cho sự phát triển của đất nước.


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Câu hỏi: Trận Điện Biên Phủ diễn ra trong thời gian nào?
Re-ranking 50 results...
Thời gian: 11.25s
Trả lời: Trận Điện Biên Phủ diễn ra từ ngày 13 tháng 3 đến ngày 7 tháng 5 năm 1954. Đây là một chiến dịch lịch sử quan trọng, đánh dấu chiến thắng vang dội của quân và dân Việt Nam trước tập đoàn cứ điểm Điện Biên Phủ của quân Pháp. Chiến dịch kéo dài 57 ngày đêm, với nhiều đợt tấn công và chiến thuật linh hoạt, đã chấm dứt sự tồn tại của tập đoàn cứ điểm này và góp phần quyết định vào thắng lợi cuối cùng của cuộc kháng chiến chống Pháp.

Câu hỏi: Thời tiết Hà Nội hôm nay thế nào?


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Thời gian: 8.37s
Trả lời: Tôi rất tiếc, nhưng câu hỏi về thời tiết Hà Nội không thuộc phạm vi chuyên môn của tôi. Bạn có thể tham khảo dự báo thời tiết từ các trang web hoặc ứng dụng theo dõi thời tiết. Nếu bạn quan tâm đến lịch sử, tôi rất sẵn lòng chia sẻ về các triều đại phong kiến, các cuộc kháng chiến chống ngoại xâm, hay thời kỳ Đổi mới. Bạn muốn tìm hiểu về chủ đề nào?


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Câu hỏi: Nhà Trần có dùng điện thoại không?
Re-ranking 50 results...
Thời gian: 12.06s
Trả lời: Câu hỏi này đề cập đến một yếu tố KHÔNG CÓ THẬT trong lịch sử Việt Nam. Thời kỳ nhà Trần (thế kỷ XIII-XIV) thuộc thời đại trước công nghệ điện thoại. Công nghệ này chỉ mới xuất hiện vào cuối thế kỷ XIX, đầu thế kỷ XX ở châu Âu và sau đó lan rộng ra toàn cầu. Trong thời nhà Trần, người ta dùng thư tín, tín ngỏ, và các phương tiện truyền tin truyền thống khác để liên lạc. Ví dụ, các sứ giả, thư ký, và các người giao dịch thường sử dụng thư từ hoặc các tín hiệu để thông báo tin tức.

Test hoàn tất
